In [22]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-24")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

In [23]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

**Problem 1** | Easy

Top order per customer

For each customer, find their single highest-value order (by unit_price). Return customer_id, order_id, unit_price.

In [24]:
window_spec1=Window.partitionBy('customer_id').orderBy(F.col('unit_price').desc())
orders_df1=orders_df.withColumn('rank',
                                F.row_number().over(window_spec1)).filter(F.col('rank')==1).\
    select('customer_id','order_id','unit_price')
orders_df1.show(5,truncate=False)

+-----------+--------+----------+
|customer_id|order_id|unit_price|
+-----------+--------+----------+
|C001       |O0001   |1299.99   |
|C002       |O0072   |1299.99   |
|C003       |O0098   |699.99    |
|C004       |O0024   |1299.99   |
|C005       |O0075   |599.99    |
+-----------+--------+----------+
only showing top 5 rows


**Problem 2** | Easy

Second highest order value per region

For each region, find the order with the second highest unit_price. Handle ties carefully — think about whether rank() or dense_rank() is correct here.

In [25]:
window_spec2=Window.partitionBy('region').orderBy(F.col('unit_price').desc())
orders_df2=orders_df.withColumn('rank',
                                F.dense_rank().over(window_spec2)).filter(F.col('rank')==2).\
    select('region','order_id','unit_price')
orders_df2.show(5,truncate=False)

+-------+--------+----------+
|region |order_id|unit_price|
+-------+--------+----------+
|East   |O0015   |699.99    |
|Midwest|O0098   |699.99    |
|South  |O0056   |699.99    |
|West   |O0037   |699.99    |
|West   |O0078   |699.99    |
+-------+--------+----------+



**Problem 3** | Medium

Running total of revenue by date

Ordered by order_date, calculate a running total of revenue (unit_price * quantity) across the entire dataset (no partition). Show date and cumulative revenue.

In [26]:
spark.sparkContext.setLogLevel("ERROR")
window_spec3 = (
    Window
    .orderBy(F.col('order_date').asc(),F.col('order_id').asc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

orders_df3 = orders_df.withColumn(
    "cumulative_revenue",
    F.sum(F.col('unit_price') * F.col('quantity')).over(window_spec3)
)
orders_df3.select('order_date','cumulative_revenue').show(5,truncate=False)

+----------+------------------+
|order_date|cumulative_revenue|
+----------+------------------+
|2023-01-05|2599.98           |
|2023-01-07|3049.9700000000003|
|2023-01-10|4449.93           |
|2023-01-12|4629.91           |
|2023-01-15|4719.88           |
+----------+------------------+
only showing top 5 rows


**Problem 4** | Medium

Month-over-month change per region

For each region, calculate total monthly revenue, then use lag() to show the previous month's revenue and the change (current minus previous) next to it.

In [27]:
window_spec = (
    Window
    .partitionBy('region')
    .orderBy('month')
)

monthly_revenue = (
    orders_df
    .withColumn('month', F.month('order_date'))
    .groupBy('region', 'month')
    .agg(
        F.sum(
            F.col('unit_price') * F.col('quantity')
        ).alias('monthly_revenue')
    )
)

result = (
    monthly_revenue
    .withColumn(
        'previous_month_revenue',
        F.lag('monthly_revenue').over(window_spec)
    )
    .withColumn(
        'change',
        F.col('monthly_revenue') -
        F.col('previous_month_revenue')
    )
    .orderBy('region', 'month')
)

result.show()

+-------+-----+------------------+----------------------+--------------------+
| region|month|   monthly_revenue|previous_month_revenue|              change|
+-------+-----+------------------+----------------------+--------------------+
|   East|    1|2799.9700000000003|                  NULL|                NULL|
|   East|    2|           2199.96|    2799.9700000000003|  -600.0100000000002|
|   East|    3|            789.97|               2199.96|            -1409.99|
|   East|    4|            629.88|                789.97| -160.09000000000003|
|   East|    5| 779.9300000000001|                629.88|  150.05000000000007|
|   East|    6|           1819.94|     779.9300000000001|             1040.01|
|   East|    7|            599.91|               1819.94| -1220.0300000000002|
|   East|    8|            569.95|                599.91| -29.959999999999923|
|   East|    9| 859.8799999999999|                569.95|  289.92999999999984|
|   East|   10|           1069.93|     859.879999999

**Problem 5** | Hard

Percentage of region's total per order

For every order, calculate what percentage of its region's total revenue that single order represents. Use a window function with no orderBy() (an unordered partition aggregate) to get the region total, then divide.

In [28]:
window_spec5 = Window.partitionBy('region')

orders_df5 = (
    orders_df
    .withColumn(
        'region_total_revenue',
        F.sum(
            F.col('unit_price') * F.col('quantity')
        ).over(window_spec5)
    )
    .withColumn(
        'percentage',
        F.round(
            (
                F.col('unit_price') * F.col('quantity')
                / F.col('region_total_revenue')
            ) * 100,
            2
        )
    )
)

orders_df5.select(
    'order_id',
    'region',
    'region_total_revenue',
    'percentage'
).show()

+--------+------+--------------------+----------+
|order_id|region|region_total_revenue|percentage|
+--------+------+--------------------+----------+
|   O0001|  East|  12119.319999999994|     21.45|
|   O0006|  East|  12119.319999999994|      1.65|
|   O0012|  East|  12119.319999999994|      4.95|
|   O0015|  East|  12119.319999999994|      5.78|
|   O0020|  East|  12119.319999999994|      7.43|
|   O0021|  East|  12119.319999999994|      0.74|
|   O0026|  East|  12119.319999999994|      5.78|
|   O0032|  East|  12119.319999999994|      1.48|
|   O0035|  East|  12119.319999999994|      2.23|
|   O0040|  East|  12119.319999999994|      1.48|
|   O0043|  East|  12119.319999999994|      1.65|
|   O0045|  East|  12119.319999999994|      2.97|
|   O0046|  East|  12119.319999999994|      1.82|
|   O0051|  East|  12119.319999999994|     10.73|
|   O0057|  East|  12119.319999999994|      2.64|
|   O0060|  East|  12119.319999999994|      1.65|
|   O0065|  East|  12119.319999999994|      1.82|


**Problem 6** | Hard

Detect consecutive order value increases

For each customer ordered by order_date, flag orders where the unit_price is strictly higher than the previous order's price. Then count, per customer, how many consecutive "increase streaks" of length 2 or more occurred.

In [29]:
window_spec = (
    Window
    .partitionBy('customer_id')
    .orderBy(
        F.col('order_date').asc(),
        F.col('order_id').asc()
    )
)

orders_df6 = (
    orders_df
    .withColumn(
        'previous_price',
        F.lag('unit_price').over(window_spec)
    )
    .withColumn(
        'is_increase',
        F.when(
            F.col('unit_price') > F.col('previous_price'),
            1
        ).otherwise(0)
    )
)

orders_df6 = orders_df6.withColumn(
    'streak_group',
    F.sum(
        F.when(F.col('is_increase') == 0, 1).otherwise(0)
    ).over(
        window_spec.rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow
        )
    )
)

streaks = (
    orders_df6
    .filter(F.col('is_increase') == 1)
    .groupBy('customer_id', 'streak_group')
    .agg(
        F.count('*').alias('streak_length')
    )
)


result = (
    streaks
    .filter(F.col('streak_length') >= 2)
    .groupBy('customer_id')
    .agg(
        F.count('*').alias('increase_streaks')
    )
    .orderBy('customer_id')
)

result.show()

+-----------+----------------+
|customer_id|increase_streaks|
+-----------+----------------+
|       C001|               1|
|       C002|               1|
|       C003|               1|
|       C006|               1|
|       C008|               1|
|       C010|               1|
|       C011|               1|
|       C012|               1|
|       C013|               1|
|       C016|               1|
|       C018|               1|
+-----------+----------------+



In [31]:
spark.stop()